⚡ Intermédiaire | ⏱ 60 min | 🔑 diagramme de classes, relations, multiplicités, Mermaid, SRP, dataclass, Strategy, Factory

# 09 - UML et conception

À la fin : tu sais dessiner et lire un diagramme de classes, passer du diagramme au code et inversement, découper proprement les responsabilités, et reconnaître deux patrons de conception courants.


## 1. UML : le diagramme de classes

UML (*Unified Modeling Language*) est une notation graphique pour **concevoir avant de coder** et **communiquer** une structure. De ses 14 diagrammes, le **diagramme de classes** est celui qu'on utilise au quotidien. Une classe = une boîte à trois compartiments : nom, attributs, méthodes.

```
┌──────────────────────────────────┐
│          CompteBancaire          │
├──────────────────────────────────┤
│ + titulaire: str                 │
│ - solde: float                   │
│ + TAUX: float = 0.02             │  ← souligné : attribut de classe
├──────────────────────────────────┤
│ + deposer(montant: float)        │
│ + retirer(montant: float): bool  │
│ - enregistrer(op: str)           │
└──────────────────────────────────┘
```

| Symbole | Visibilité | En Python |
|---|---|---|
| `+` | public | `nom` |
| `#` | protégé | `_nom` |
| `-` | privé | `_nom` ou `__nom` |
| souligné | statique (de classe) | attribut de classe, `@classmethod`, `@staticmethod` |


Dans ce dépôt on dessine en **Mermaid** : du texte, versionné avec le code, rendu par VS Code, JupyterLab et GitHub. Le bloc ci-dessous est du Markdown ; si ton éditeur ne le rend pas, colle-le sur [mermaid.live](https://mermaid.live).

```mermaid
classDiagram
    class CompteBancaire {
        +str titulaire
        -float solde
        +float TAUX$
        +deposer(montant)
        +retirer(montant) bool
        -enregistrer(op)
    }
```


In [ ]:
# Le code qui correspond au diagramme ci-dessus
class CompteBancaire:
    TAUX = 0.02

    def __init__(self, titulaire, solde=0.0):
        self.titulaire = titulaire
        self._solde = solde

    def deposer(self, montant):
        self._solde += montant
        self._enregistrer("dépôt")

    def retirer(self, montant):
        if montant > self._solde:
            return False
        self._solde -= montant
        self._enregistrer("retrait")
        return True

    def _enregistrer(self, op):
        print("opération :", op)

CompteBancaire("Ana", 10).deposer(5)


## 2. Relations entre classes

| Relation | Phrase | UML | Mermaid | Code |
|---|---|---|---|---|
| Héritage | « un Chat **est un** Animal » | flèche à triangle vide vers la mère | `Animal <\|-- Chat` | `class Chat(Animal)` |
| Composition | « une Voiture **a un** Moteur, créé avec elle » | losange plein côté tout | `Voiture *-- Moteur` | `self.moteur = Moteur()` dans `__init__` |
| Agrégation | « une Équipe **a des** Joueurs, qui existent sans elle » | losange vide côté tout | `Equipe o-- Joueur` | `self.joueurs = joueurs` reçus en paramètre |
| Association | « un Prof **enseigne** des Cours » | trait simple | `Prof -- Cours` | un attribut qui référence l'autre objet |
| Dépendance | « Facture **utilise** Tva » | flèche pointillée | `Facture ..> Tva` | paramètre ou appel ponctuel |
| Réalisation | « Carre **implémente** Forme » | flèche pointillée à triangle vide | `Forme <\|.. Carre` | `class Carre(Forme)` avec `Forme(ABC)` |

**Multiplicités** aux extrémités : `1`, `0..1`, `*` (ou `0..*`), `1..*`, `2..4`. Elles se lisent du côté opposé : « un Membre emprunte 0 à 3 Livres ».

```mermaid
classDiagram
    Personne <|-- Membre
    Bibliotheque o-- "0..*" Livre : possède
    Membre "0..1" -- "0..3" Livre : emprunte
    class Livre {
        +str titre
        +bool disponible
    }
    class Membre {
        +emprunter(livre) bool
        +rendre(livre)
    }
```


In [ ]:
# Du diagramme au code : une classe par boîte, une relation par attribut
class Personne:
    def __init__(self, nom):
        self.nom = nom

class Livre:
    def __init__(self, titre):
        self.titre = titre
        self.disponible = True

class Membre(Personne):                         # héritage
    def __init__(self, nom):
        super().__init__(nom)
        self.emprunts = []                      # association 0..3

    def emprunter(self, livre):
        if not livre.disponible or len(self.emprunts) >= 3:
            return False
        livre.disponible = False
        self.emprunts.append(livre)
        return True

    def rendre(self, livre):
        self.emprunts.remove(livre)
        livre.disponible = True

class Bibliotheque:
    def __init__(self, livres):
        self.livres = list(livres)              # agrégation : les livres préexistent

dune = Livre("Dune")
ana = Membre("Ana")
biblio = Bibliotheque([dune])
print(ana.emprunter(dune), ana.emprunter(dune), dune.disponible)


## 3. Du cahier des charges au diagramme

Méthode en quatre passes sur le texte du besoin :

1. Les **noms** deviennent des classes candidates : *livre*, *membre*, *bibliothèque*.
2. Les **données** attachées à un nom deviennent des attributs : *titre*, *disponible*.
3. Les **verbes** deviennent des méthodes : *emprunter*, *rendre*.
4. Les **phrases de liaison** (« a », « est un », « utilise ») deviennent des relations, avec leurs multiplicités : « un membre peut emprunter jusqu'à trois livres ».

Puis on élague : un nom sans comportement ni relation est souvent un simple attribut (*ISBN* est un attribut de Livre, pas une classe).


## 4. Une classe, une responsabilité

Principe de responsabilité unique (*SRP*) : une classe doit avoir **une seule raison de changer**. `Utilisateur` qui stocke des données, envoie des mails et écrit en base changera pour trois raisons différentes et deviendra ingérable. Signes d'alerte : classe de 500 lignes, nom en « Gestionnaire » ou « Système », méthodes sans rapport entre elles.


In [ ]:
# ❌ Une classe qui fait tout : données, email, base de données
class Utilisateur:
    def __init__(self, nom, email):
        self.nom, self.email = nom, email

    def envoyer_mail(self, texte):
        return f"mail à {self.email} : {texte}"

    def sauvegarder(self):
        return f"INSERT INTO users VALUES ('{self.nom}')"

u = Utilisateur("Ana", "ana@ex.fr")
print(u.envoyer_mail("bienvenue"), "|", u.sauvegarder())


In [ ]:
# ✅ Séparer : chaque classe change pour une seule raison
class Utilisateur:
    def __init__(self, nom, email):
        self.nom, self.email = nom, email

class ServiceMail:
    def envoyer(self, utilisateur, texte):
        return f"mail à {utilisateur.email} : {texte}"

class DepotUtilisateurs:
    def sauvegarder(self, utilisateur):
        return f"INSERT INTO users VALUES ('{utilisateur.nom}')"

u = Utilisateur("Ana", "ana@ex.fr")
print(ServiceMail().envoyer(u, "bienvenue"), "|", DepotUtilisateurs().sauvegarder(u))


## 5. Moins de code : `@dataclass`

Pour une classe qui sert surtout à **porter des données**, `@dataclass` génère `__init__`, `__repr__` et `__eq__` à partir des annotations de type. On peut toujours ajouter des méthodes. `frozen=True` rend l'objet immuable, donc hachable.


In [ ]:
# @dataclass : __init__, __repr__ et __eq__ générés à partir des annotations
from dataclasses import dataclass, field

@dataclass
class Livre:
    titre: str
    auteur: str
    annee: int | None = None
    tags: list[str] = field(default_factory=list)   # jamais de liste mutable par défaut

    def age(self, annee_courante=2026):
        return None if self.annee is None else annee_courante - self.annee

l = Livre("Dune", "Herbert", 1965)
print(l, l == Livre("Dune", "Herbert", 1965), l.age())


In [ ]:
# frozen=True : immuable, donc utilisable comme clé de dict ou dans un set
@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1, 2)
try:
    p.x = 5
except Exception as e:
    print(type(e).__name__, ":", e)
print({p, Point(1, 2)})


## 6. Patrons de conception : deux à reconnaître

Un **design pattern** est une solution nommée à un problème récurrent de conception. Le catalogue classique (*Gang of Four*, 1994) en compte 23 ; en connaître le vocabulaire aide à lire du code et à discuter en équipe. Deux reviennent tout le temps en data et s'appuient sur ce que tu viens de voir.

| Patron | Problème | Solution | Déjà vu dans |
|---|---|---|---|
| **Strategy** | changer d'algorithme sans toucher au code qui l'utilise | une classe par algorithme, même interface, injectée par composition | notebook 05, `Utilisateur(canal)` |
| **Factory** | créer le bon objet selon une donnée (extension de fichier, canal) sans répandre des `if` partout | une fonction ou méthode de classe qui centralise la création | notebook 01, `Date.depuis_iso` |


In [ ]:
# Strategy : la commande ne sait pas comment on calcule les frais, elle délègue
from abc import ABC, abstractmethod

class Livraison(ABC):
    @abstractmethod
    def frais(self, poids):
        ...

class Standard(Livraison):
    def frais(self, poids):
        return 5 + 2 * poids

class Express(Livraison):
    def frais(self, poids):
        return 15 + 5 * poids

class Commande:
    def __init__(self, poids, livraison):
        self.poids = poids
        self.livraison = livraison

    def total_frais(self):
        return self.livraison.frais(self.poids)

cmd = Commande(3, Standard())
print(cmd.total_frais())
cmd.livraison = Express()
print(cmd.total_frais())


In [ ]:
# Factory : un seul endroit décide quelle classe instancier
class LecteurCSV:
    def lire(self):
        return "lecture CSV"

class LecteurJSON:
    def lire(self):
        return "lecture JSON"

LECTEURS = {"csv": LecteurCSV, "json": LecteurJSON}

def creer_lecteur(chemin):
    extension = chemin.rsplit(".", 1)[-1].lower()
    try:
        return LECTEURS[extension]()
    except KeyError:
        raise ValueError(f"format non géré : {extension}") from None

print(creer_lecteur("ventes.CSV").lire(), "|", creer_lecteur("conf.json").lire())


Quatre autres patrons à reconnaître de nom, détaillés dans la cheatsheet : **Observer** (prévenir des abonnés d'un changement), **Decorator** (envelopper un objet pour lui ajouter un comportement), **Adapter** (faire parler deux interfaces incompatibles), **Singleton** (une seule instance, à utiliser avec parcimonie).


## ⚠️ Pièges

- Diagramme exhaustif de 40 classes : illisible, et faux dans deux semaines. Diagrammer ce qui aide à décider ; le code reste la référence.
- Sens des flèches : l'héritage pointe vers la **mère**, le losange est du côté du **tout**.
- Classe « Gestionnaire » ou « Système » qui fait tout.
- Appliquer un patron « au cas où » : un `Calculateur` avec Factory et Strategy pour additionner deux nombres. Commencer simple, refactorer vers un patron quand le besoin apparaît.
- `@dataclass` avec `tags: list = []` : Python refuse (`mutable default`). Utiliser `field(default_factory=list)`.


In [ ]:
# BUG classique : valeur mutable par défaut dans une dataclass
from dataclasses import dataclass

try:
    @dataclass
    class Panier:
        articles: list = []
except ValueError as e:
    print("ValueError :", e)
